# Phase 7 — feedback-consistent constrained decoding

Retrain `tree_salet_endgame` (identical Phase 6 config and dataset), then
compare four decoders on the same 246 held-out answers:

| decoder | admissible set |
|---|---|
| `unconstrained` | anything the model emits |
| `legal` | the 12,972 legal guesses |
| `consistent` | legal **and** consistent with all feedback, every turn |
| `adaptive` | consistent only once the admissible set is small |

## Where Phase 6 left this

| | mean | fail | solved |
|---|---:|---:|---:|
| `tree_salet` [noban] | 5.6870 | 56.1% | 108 |
| `tree_salet_endgame` [noban] | 5.6992 | 56.1% | 108 |
| `tree_salet_endgame` [ban] | 5.4797 | 44.7% | 136 |
| classical `random` | 4.0203 | 0.8% | 244 |

The endgame data changed games not at all; banning repeats did the work. But it
*did* teach something real: on words never trained as targets, k=1 accuracy went
15.25% → 25.42% (paired McNemar, 6 gained / 0 lost, p = 0.031) with zero change
on trained words.

The model reaches ~1.2 candidates by turn 3 and then cannot name the word. Two
measurements say the decoder is the lever:

* **hard-mode violations 31%** — a third of guesses contradict feedback already
  received
* at k=1, a **median of 2** legal words are consistent with the revealed
  feedback, against a pool of 12,972

## What the consistent decoder does

A word is admissible iff it would have produced **exactly** the feedback already
observed, for every guess so far:

```python
allowed = [w for w in allowed if feedback_code(guess, w) == observed_code]
```

Two sets are easy to conflate, and the difference is the entire justification:

| set | pool | uses the answer list? |
|---|---|---|
| candidate set | 2,315 answers | **yes** — privileged, never used |
| **hard-mode set** | 12,972 legal guesses | no — a function of the prompt |

The model never sees the admissible set, its size, the answer, or the answer
list. This is a decoder-side restriction of the same kind as the legal-word
constraint — it is what hard mode enforces and what a human sees on their board.

## Read forced and model-chosen separately

Measured on the expert's own games: at k=1 states the filter **alone** leaves
exactly one word **38.9%** of the time. In those the decoder has solved the
game and the model contributed nothing.

So every decision is tagged:

* `forced` — one admissible word. **Not** a model decision.
* `model_chosen` — several admissible; the model ranked them.

A headline that merged the two would let a decoder win masquerade as a model
win. The unfiltered k=1 probe stays as the measure of what the model knows.

---

## How to run

**STEP 1** — Accelerator: **GPU T4 x2**.

**STEP 2** — Add Input: the **v2** SFT package (with
`data/tree_salet_endgame/`). Optionally also an adapters dataset — if it
contains `tree_salet_endgame`, training is skipped entirely.

**STEP 3** — Run All. ~2.5 h from scratch, ~1 h if the adapter already exists.

**STEP 4 — SAVE THE ADAPTER THE MOMENT SECTION 7 FINISHES.** The notebook
stops and tells you how. Do not skip it; this is why Phase 6 had to be retrained.

**STEP 5** — Download `wordle_phase7_results.zip` (small, no weights).

---
# 1. Environment

In [ ]:
import os, sys, json, time, math, random, subprocess, platform, shutil
import importlib
from collections import Counter

def _pip(p):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", p], check=False)

try:
    import torch
except ImportError:
    _pip("torch"); import torch
for mod, pkg in [("transformers", "transformers>=4.44"), ("peft", "peft>=0.11"),
                 ("accelerate", "accelerate>=0.30")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        _pip(pkg)
import torch, transformers, peft, accelerate
import numpy as np

def fix_torchao_peft_conflict():
    """peft's torchao probe RAISES on an outdated torchao instead of returning
    False, which kills get_peft_model. Kaggle ships an old one; we never use it."""
    try:
        import peft.import_utils as piu
    except Exception as e:
        return f"unavailable ({e})"
    try:
        piu.is_torchao_available(); return "no conflict"
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                    "torchao"], check=False)
    importlib.invalidate_caches()
    try:
        piu.is_torchao_available(); return "resolved: uninstalled torchao"
    except ImportError:
        pass
    piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as t
        t.is_torchao_available = lambda *a, **k: False
    except Exception:
        pass
    return "resolved: patched probe"

_FIX = fix_torchao_peft_conflict()
print("=" * 66); print("ENVIRONMENT"); print("=" * 66)
for k, v in [("python", platform.python_version()), ("torch", torch.__version__),
             ("transformers", transformers.__version__), ("peft", peft.__version__),
             ("numpy", np.__version__), ("torchao fix", _FIX)]:
    print(f"{k:<14} {v}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"\nGPU {p.name}  {p.total_memory/2**30:.1f} GiB  x{torch.cuda.device_count()}")
else:
    print("\n!! NO GPU !! Session options -> Accelerator -> GPU T4 x2")

---
# 2. Configuration

Training config is byte-identical to Phase 6 so the retrain is a reproduction.

In [ ]:
# ============================ EDIT THIS =====================================
DATASET_DIR  = None    # auto-detect; must contain data/tree_salet_endgame/
PREV_RUN_DIR = None    # optional: adapters dataset. If it has
                       # tree_salet_endgame, training is SKIPPED.
# ============================================================================

FORCE_RETRAIN  = False      # True only to deliberately overwrite an adapter
RUN_EVALUATION = True
RUN_BASELINES  = True
RUN_K1_PROBE   = True
RUN_BASE_CONTROL = False    # base Qwen: ~4h in legal mode (pruning cannot help
                            # an untrained model). Leave False unless you have
                            # the session budget; Phase 5 already measured it
                            # at 7.0000 / 100% fail in both modes.

# ---- identical to Phase 6 --------------------------------------------------
MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_NAME = "tree_salet_endgame"
USE_NATURAL, USE_ENDGAME, ENDGAME_REPEAT = True, True, 1
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
LEARNING_RATE, NUM_EPOCHS = 2e-4, 2
PER_DEVICE_BS, GRAD_ACCUM = 4, 4
MAX_SEQ_LEN, WARMUP_RATIO, WEIGHT_DECAY = 640, 0.03, 0.0
LR_SCHEDULER, FP16, GRAD_CHECKPOINT = "cosine", True, True
LOGGING_STEPS, SAVE_STEPS, SAVE_TOTAL_LIMIT = 25, 200, 2
SEED = 20260817

# ---- evaluation ------------------------------------------------------------
MAX_GUESSES, GEN_MAX_NEW_TOKENS, EVAL_BATCH = 6, 8, 16
CONSTRAINED_CHUNK, CONSTRAINED_PRUNE, LENGTH_NORMALISE = 512, True, False

# Applying the consistency filter at EVERY turn is not obviously right, and the
# training data says so. The expert's own target is feedback-inconsistent most
# of the time early on:
#
#     turn 2:  40.6% consistent   <- the expert PROBES: it deliberately plays a
#     turn 3:  91.8% consistent      word that cannot be the answer, to split
#     turn 4: 100.0% consistent      the space
#
# An always-on filter therefore forbids the expert's own turn-2 policy in ~59%
# of games. That is the known reason hard mode scores worse than free mode. So
# we also test an ADAPTIVE filter: probe freely while uncertainty is high, become
# consistent once it is low. The trigger is the admissible-set size, which is
# derivable from the prompt like the filter itself (measured medians: turn 2
# ~211, turn 3 ~7, turn 4 ~2, so 50 separates probing from resolving).
ADAPTIVE_THRESHOLD = 50

# (decoder, ban_repeats). Banning is reported separately from the decoder so the
# two effects never get merged.
EVAL_MATRIX = [
    ("unconstrained", False),
    ("legal",         False),
    ("legal",         True),
    ("consistent",    False),
    ("consistent",    True),
    ("adaptive",      True),
]

K1_MAX_STATES = 80

PHASE6 = {"endgame_legal_noban": 5.6992, "endgame_legal_ban": 5.4797,
          "salet_legal_noban": 5.6870, "k1_top1": 27.5, "k1_median_rank": 4.0,
          "k1_seen": 33.33, "k1_unseen": 25.42, "hard_mode_viol": 31.18,
          "classical_random": 4.0203, "classical_entropy": 3.4431}

WORK_DIR     = "/kaggle/working/wordle_phase7"
RESULTS_ROOT = "/kaggle/working/results_phase7"
RESULTS_ZIP  = "/kaggle/working/wordle_phase7_results.zip"
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(RESULTS_ROOT, exist_ok=True)

def set_seed_everywhere(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)

set_seed_everywhere()
_ngpu = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"adapter {ADAPTER_NAME}")
print(f"effective batch {PER_DEVICE_BS} x {GRAD_ACCUM} x {_ngpu} GPU(s) = "
      f"{PER_DEVICE_BS*GRAD_ACCUM*_ngpu}   (Phase 6 ran the same way)")
print(f"eval matrix: {EVAL_MATRIX}")

---
# 3. Dataset

In [ ]:
import glob

REQUIRED_FILES = [
    "sft_package/data/tree_salet/train.jsonl",
    "sft_package/data/tree_salet_endgame/train.jsonl",
    "sft_package/eval/val_answers.jsonl",
    "sft_package/eval/train_answers.jsonl",
    "code/wordle_solver.py", "code/tree_search.py",
    "code/generate_trajectories.py",
    "artifacts/answers.txt", "artifacts/valid_guesses.txt",
    "artifacts/feedback_matrix.npy",
]

def _has_all(d):
    try:
        return all(os.path.exists(os.path.join(d, f)) for f in REQUIRED_FILES)
    except OSError:
        return False

def _search(root, max_depth=7):
    if not os.path.isdir(root):
        return None
    for depth in range(0, 6):
        pat = os.path.join(root, *(["*"] * depth)) if depth else root
        for d in sorted(glob.glob(pat)):
            if os.path.isdir(d) and _has_all(d):
                return d
    base = os.path.abspath(root).rstrip(os.sep).count(os.sep)
    best = None
    for dp, dn, _ in os.walk(root):
        if os.path.abspath(dp).rstrip(os.sep).count(os.sep) - base > max_depth:
            dn[:] = []; continue
        dn[:] = [d for d in dn if not d.startswith(".")]
        if _has_all(dp):
            best = dp if best is None or len(dp) < len(best) else best
            dn[:] = []
    return best

def locate_dataset(explicit=None):
    for c in ([explicit, os.path.join(explicit or "", "kaggle_upload")]
              if explicit else []):
        if _has_all(c):
            return c
    for root in ([explicit] if explicit else []) + ["/kaggle/input", ".",
                                                    "/kaggle/working"]:
        hit = _search(root)
        if hit:
            return hit
    near = [dp for dp, _, _ in os.walk("/kaggle/input")
            if os.path.exists(os.path.join(dp, "sft_package/data/tree_salet/train.jsonl"))]
    raise FileNotFoundError("\n".join(
        ["Dataset not found."] +
        (["Found a package WITHOUT the endgame file:"] + [f"  {n}" for n in near]
         + ["", "Missing: sft_package/data/tree_salet_endgame/train.jsonl",
            "Attach the v2 package (rebuilt after Phase 6)."] if near
         else ["Nothing resembling the SFT package under /kaggle/input."])))

DATA_ROOT = locate_dataset(DATASET_DIR)
SFT_DIR = os.path.join(DATA_ROOT, "sft_package")
ARTIFACTS = os.path.join(DATA_ROOT, "artifacts")
CODE_DIR = os.path.join(DATA_ROOT, "code")
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print(f"dataset root: {DATA_ROOT}")

from wordle_solver import (load_artifacts, SolverConfig, make_solver, play_game,
                           feedback_code, code_to_pattern, ALL_GREEN)
from generate_trajectories import derive_constraints, render_prompt

BUNDLE = load_artifacts(ARTIFACTS, mmap=True)
VOCAB = BUNDLE.vocab
LEGAL_LOWER = [g.lower() for g in VOCAB.guesses]
LEGAL_GUESSES = set(w.upper() for w in LEGAL_LOWER)
VAL_ANSWERS = [json.loads(l)["answer"].upper()
               for l in open(os.path.join(SFT_DIR, "eval/val_answers.jsonl"),
                             encoding="utf-8")]
assert len(LEGAL_GUESSES) == 12972 and len(VAL_ANSWERS) == 246
print(f"legal {len(LEGAL_GUESSES)}   held-out {len(VAL_ANSWERS)}")

---
# 4. Model, mix, and training

Training is **skipped** if an adapter already exists. That is the resumability
guarantee: re-running this notebook after a session dies costs an evaluation,
not a retrain.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.utils.data import Dataset
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import Trainer, TrainingArguments

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

def load_base_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map=None,
        trust_remote_code=True)
    m.config.use_cache = False
    return m

def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

NATURAL = load_jsonl(os.path.join(SFT_DIR, "data/tree_salet/train.jsonl"))
ENDGAME = load_jsonl(os.path.join(SFT_DIR, "data/tree_salet_endgame/train.jsonl"))
ROWS = (NATURAL if USE_NATURAL else []) + (ENDGAME * ENDGAME_REPEAT if USE_ENDGAME else [])
random.Random(SEED).shuffle(ROWS)

k1 = [r for r in ROWS if r["meta"]["n_candidates"] == 1]
per = Counter(r["completion"] for r in k1)
DATA_STATS = {"n_rows": len(ROWS), "natural": len(NATURAL), "endgame": len(ENDGAME),
              "k1_rows": len(k1), "k1_words": len(per),
              "k1_paths_per_word": round(len(k1)/max(len(per), 1), 3),
              "turn1_pct": round(100*sum(1 for r in ROWS if r["meta"]["turn"] == 1)/len(ROWS), 2)}
print(f"mix: {DATA_STATS['n_rows']} rows  k=1 paths/word "
      f"{DATA_STATS['k1_paths_per_word']}  turn-1 {DATA_STATS['turn1_pct']}%")
assert abs(DATA_STATS["k1_paths_per_word"] - 3.752) < 0.01, \
    "mix differs from Phase 6 - the retrain would not be a reproduction"
print("mix matches Phase 6 exactly  OK")

class WordleSFTDataset(Dataset):
    def __init__(self, rows, tok, max_len=MAX_SEQ_LEN):
        self.rows, self.tok, self.max_len = rows, tok, max_len
        self.n_truncated = 0; self._c = {}
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if i in self._c: return self._c[i]
        r = self.rows[i]
        p = self.tok(r["prompt"], add_special_tokens=False)["input_ids"]
        c = self.tok(" " + r["completion"], add_special_tokens=False)["input_ids"]
        c = c + [self.tok.eos_token_id]
        keep = self.max_len - len(c)
        if len(p) > keep:
            p = p[-keep:]; self.n_truncated += 1
        it = {"input_ids": p + c, "labels": [-100]*len(p) + c,
              "attention_mask": [1]*(len(p)+len(c))}
        self._c[i] = it; return it

def collate(batch, pad_id):
    n = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        d = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad_id]*d)
        out["labels"].append(b["labels"] + [-100]*d)
        out["attention_mask"].append(b["attention_mask"] + [0]*d)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

def adapter_path(name):
    for base in (WORK_DIR, PREV_RUN_DIR or ""):
        p = os.path.join(base, name)
        if os.path.exists(os.path.join(p, "adapter_config.json")):
            return p
    return None

TRAIN_CONFIG = {}
existing = adapter_path(ADAPTER_NAME)
out_dir = os.path.join(WORK_DIR, ADAPTER_NAME)

if existing and not FORCE_RETRAIN:
    print(f"\nADAPTER FOUND at {existing} - training SKIPPED")
    cp = os.path.join(existing, "training_config.json")
    TRAIN_CONFIG = json.load(open(cp)) if os.path.exists(cp) else {"name": ADAPTER_NAME}
else:
    print(f"\n{'='*66}\nTRAINING {ADAPTER_NAME} on {len(ROWS)} rows\n{'='*66}")
    set_seed_everywhere()
    fix_torchao_peft_conflict()
    base = load_base_model()
    model = get_peft_model(base, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"))
    n = 0
    for _, prm in model.named_parameters():
        if prm.requires_grad and prm.dtype == torch.float16:
            prm.data = prm.data.float(); n += 1
    if n: print(f"  cast {n} trainable tensors fp16 -> fp32 (amp requirement)")
    model.print_trainable_parameters()
    ds = WordleSFTDataset(ROWS, TOKENIZER)
    trainer = Trainer(
        model=model,
        args=TrainingArguments(
            output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=PER_DEVICE_BS,
            gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LEARNING_RATE,
            lr_scheduler_type=LR_SCHEDULER, warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY, fp16=FP16, bf16=False,
            gradient_checkpointing=GRAD_CHECKPOINT, logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS, save_total_limit=SAVE_TOTAL_LIMIT,
            save_strategy="steps", report_to=[], seed=SEED, data_seed=SEED,
            optim="adamw_torch", max_grad_norm=1.0, dataloader_num_workers=2,
            remove_unused_columns=False, disable_tqdm=False),
        train_dataset=ds,
        data_collator=lambda b: collate(b, TOKENIZER.pad_token_id))
    t0 = time.perf_counter(); trainer.train(); secs = time.perf_counter()-t0
    trainer.save_model(out_dir)
    hist = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    TRAIN_CONFIG = {"name": ADAPTER_NAME, "n_train_examples": len(ROWS),
                    "n_truncated": ds.n_truncated, "training_seconds": round(secs, 1),
                    "global_steps": trainer.state.global_step,
                    "first_loss": hist[0] if hist else None,
                    "final_loss": hist[-1] if hist else None,
                    "data_stats": DATA_STATS, "seed": SEED,
                    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
    json.dump(TRAIN_CONFIG, open(os.path.join(out_dir, "training_config.json"),
                                 "w", encoding="utf-8"), indent=2, default=str)
    print(f"\ntrained in {secs/60:.1f} min, {trainer.state.global_step} steps, "
          f"loss {hist[0]:.3f} -> {hist[-1]:.4f}")
    del model, trainer; torch.cuda.empty_cache()

ADAPTER_DIR = adapter_path(ADAPTER_NAME)
assert ADAPTER_DIR, "no adapter available"
print(f"\nadapter: {ADAPTER_DIR}")

---
# 5. ⚠ SAVE THE ADAPTER NOW

Phase 6's weights were lost to a session teardown and had to be retrained. Do
this before running anything else.

In [ ]:
print("=" * 70)
print("SAVE THE ADAPTER BEFORE CONTINUING")
print("=" * 70)
print(f"  path: {os.path.join(WORK_DIR, ADAPTER_NAME)}")
tot = 0
for r, _, fs in os.walk(WORK_DIR):
    for f in fs:
        tot += os.path.getsize(os.path.join(r, f))
print(f"  size: {tot/2**20:.1f} MiB")
print("""
  1. Right sidebar -> Output
  2. New Dataset  ->  save  /kaggle/working/wordle_phase7
  3. Title it  wordle-phase7-adapter

Next session: Add Input that dataset and set

    PREV_RUN_DIR = "/kaggle/input/.../wordle_phase7"

and section 4 will skip training entirely.

/kaggle/working does NOT survive a session ending. This is the step that was
missed in Phase 6.""")

---
# 6. The decoders

`constrained_decode.py` verbatim, then the three-way harness. The
`HardModeFilter` docstring states precisely why the consistent set is not
privileged information.

In [ ]:
"""
constrained_decode.py — exact argmax over a fixed legal-word set.

The question this answers is

    "Which of these 12,972 legal Wordle words should I play?"

not

    "Generate arbitrary text and see whether it happens to be a word."

Nothing is filtered after the fact. The model never emits free text at all in
constrained mode: every legal word is *scored*, and the highest-scoring one is
played.

--------------------------------------------------------------------------
Definition of the score
--------------------------------------------------------------------------
For a prompt `p` and a legal word `w`, tokenize exactly as the SFT data did --
`" " + w` -- and append EOS. Then

    score(w) = log P(w | p) = sum_i log P(t_i | p, t_0..t_{i-1})

summed over the word's tokens **and the EOS token**.

Including EOS matters. Without it, a word whose token sequence is a prefix of a
longer word's is scored on a strictly smaller set of constraints and is
systematically over-ranked; with EOS the scores are log-probabilities of
complete strings, so they are directly comparable across different token
lengths. No length normalisation is applied: `score(w)` is exactly the
probability the model assigns to playing `w`, which is the quantity we want to
argmax. (`length_normalise=True` is available for a sensitivity check but is a
heuristic, not the default.)

--------------------------------------------------------------------------
How it is computed
--------------------------------------------------------------------------
Naively this is 12,972 forward passes per turn. Instead:

1. The prompt is run **once** with `use_cache=True`, giving a KV cache and the
   next-token distribution `lp0` over the whole vocabulary.
2. `lp0` already gives the first-token log-probability of every legal word, for
   free -- no forward pass.
3. The remaining tokens are scored by teacher forcing: the prompt's KV cache is
   expanded to a batch of `chunk` rows and the padded `[chunk, L]` word-token
   matrix is pushed through in one forward pass. Causal masking makes the
   right-hand padding inert.

Exact branch-and-bound pruning (`prune=True`, on by default and *exact*):
every per-token log-probability is <= 0, so

    score(w) <= lp0[first_token(w)]

is a valid upper bound. Words are visited in descending order of that bound;
once the best fully-scored word beats the bound of every unvisited word, no
unvisited word can win and the scan stops. The returned argmax is identical to
scoring all 12,972 -- `verify_against_full()` asserts exactly that.

--------------------------------------------------------------------------
What is NOT done here
--------------------------------------------------------------------------
- The candidate-answer list is never consulted. The scorer ranks the full legal
  guess pool; it has no idea which words are still possible.
- The hidden answer is never consulted.
- Repeats are not banned by default (`banned` is opt-in), because banning them
  would be a policy change, not a vocabulary constraint.
"""

import numpy as np
import torch
import torch.nn.functional as F

NEG_INF = float("-inf")


# ---------------------------------------------------------------------------
# KV-cache compatibility
#
# transformers has moved the cache representation twice (legacy tuple ->
# DynamicCache.key_cache/value_cache -> Cache.layers). Rather than pin a
# version, read whichever layout is present and rebuild through the same one.
# `LegalWordScorer.self_test()` verifies the result numerically at runtime, so
# a layout this shim gets wrong fails loudly instead of scoring garbage.
# ---------------------------------------------------------------------------
def _cache_layers(cache):
    if hasattr(cache, "layers"):                        # transformers >= 5
        return [(lyr.keys, lyr.values) for lyr in cache.layers]
    if hasattr(cache, "key_cache"):                     # transformers 4.x
        return list(zip(cache.key_cache, cache.value_cache))
    return [(k, v) for k, v in cache]                   # legacy tuple-of-tuples


def _rebuild_cache(template, layers):
    """Rebuild a cache object of the same kind as `template` from `layers`."""
    if hasattr(template, "layers"):
        import copy
        new = copy.deepcopy(template)
        for lyr, (k, v) in zip(new.layers, layers):
            lyr.keys, lyr.values = k, v
        return new
    if hasattr(template, "key_cache"):
        from transformers.cache_utils import DynamicCache
        new = DynamicCache()
        new.key_cache = [k for k, _ in layers]
        new.value_cache = [v for _, v in layers]
        return new
    return tuple(layers)


def expand_cache(cache, n):
    """Repeat a batch-1 KV cache to batch `n` without recomputing the prompt."""
    out = []
    for k, v in _cache_layers(cache):
        assert k.shape[0] == 1, f"expected batch-1 cache, got {k.shape[0]}"
        out.append((k.expand(n, *k.shape[1:]).contiguous(),
                    v.expand(n, *v.shape[1:]).contiguous()))
    return _rebuild_cache(cache, out)


# ---------------------------------------------------------------------------
class LegalWordScorer:
    """Scores every word in a fixed legal set under a causal LM."""

    def __init__(self, tokenizer, words, device="cuda", chunk=512,
                 length_normalise=False):
        self.tok = tokenizer
        self.words = list(words)
        self.device = device
        self.chunk = chunk
        self.length_normalise = length_normalise
        self.n = len(self.words)

        eos = tokenizer.eos_token_id
        seqs = []
        for w in self.words:
            # EXACTLY the training-time tokenization: " " + WORD, then EOS.
            ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
            seqs.append(ids + [eos])
        self.max_len = max(len(s) for s in seqs)

        pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos
        tokens = np.full((self.n, self.max_len), pad, dtype=np.int64)
        mask = np.zeros((self.n, self.max_len), dtype=np.float32)
        for i, s in enumerate(seqs):
            tokens[i, :len(s)] = s
            mask[i, :len(s)] = 1.0

        self.tokens = torch.from_numpy(tokens).to(device)      # [N, L]
        self.mask = torch.from_numpy(mask).to(device)          # [N, L]
        self.lengths = torch.from_numpy(mask.sum(1)).to(device)
        self.first_tok = self.tokens[:, 0].clone()             # [N]
        self.index = {w: i for i, w in enumerate(self.words)}

    # -- internals ----------------------------------------------------------
    @torch.no_grad()
    def _prompt_pass(self, model, prompt):
        ids = self.tok(prompt, return_tensors="pt",
                       add_special_tokens=False)["input_ids"].to(self.device)
        out = model(input_ids=ids, use_cache=True)
        lp0 = F.log_softmax(out.logits[0, -1].float(), dim=-1)   # [V]
        return out.past_key_values, lp0, ids.shape[1]

    @torch.no_grad()
    def _score_rows(self, model, cache, lp0, prompt_len, rows):
        """Exact log P(w | prompt) for the word indices in `rows`."""
        idx = rows.to(self.device)
        toks = self.tokens[idx]                                  # [C, L]
        msk = self.mask[idx]
        C, L = toks.shape

        total = lp0[toks[:, 0]] * msk[:, 0]                      # token 0, free
        if L > 1:
            big = expand_cache(cache, C)
            attn = torch.ones(C, prompt_len + L, dtype=torch.long,
                              device=self.device)
            pos = torch.arange(prompt_len, prompt_len + L,
                               device=self.device).unsqueeze(0).expand(C, L)
            out = model(input_ids=toks, past_key_values=big,
                        attention_mask=attn, position_ids=pos, use_cache=False)
            # logits[:, i] predicts token i+1, so positions 1..L-1 read 0..L-2.
            for i in range(1, L):
                lp = F.log_softmax(out.logits[:, i - 1].float(), dim=-1)
                total = total + lp.gather(1, toks[:, i:i + 1]).squeeze(1) * msk[:, i]
            del out, big
        if self.length_normalise:
            total = total / msk.sum(1)
        return total

    # -- public API ---------------------------------------------------------
    @torch.no_grad()
    def score_all(self, model, prompt):
        """Score every legal word. No pruning. Returns a [N] float tensor."""
        cache, lp0, plen = self._prompt_pass(model, prompt)
        out = torch.empty(self.n, dtype=torch.float32, device=self.device)
        for s in range(0, self.n, self.chunk):
            rows = torch.arange(s, min(s + self.chunk, self.n))
            out[rows.to(self.device)] = self._score_rows(
                model, cache, lp0, plen, rows)
        return out

    @torch.no_grad()
    def argmax(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Highest-scoring legal word.

        With `prune=True` this is the *same* word `score_all().argmax()` gives;
        the bound is exact, not a heuristic. Returns
        `(word, score, n_chunks_scored, margin)`.

        `margin` is the gap to the runner-up **among words actually scored**.
        With pruning on, unvisited words are known to score below the winner but
        could sit above the runner-up, so `margin` is an upper bound on the true
        margin. It is a diagnostic, never an input to a decision.
        """
        cache, lp0, plen = self._prompt_pass(model, prompt)

        bound = lp0[self.first_tok].clone()                      # [N] upper bd
        # `ban_mask` marks words that must NOT be returned, for any reason.
        # It is applied to the bound (so they sort last and prune early) AND to
        # the scores (so one cannot win from the tail of a live chunk). Masking
        # only the bound is a real bug we shipped once: the excluded word still
        # received a genuine score and could come out on top.
        ban_mask = torch.zeros(self.n, dtype=torch.bool, device=self.device)

        if allowed_idx is not None:
            keep = torch.zeros(self.n, dtype=torch.bool, device=self.device)
            keep[torch.as_tensor(np.asarray(allowed_idx), device=self.device)] = True
            ban_mask |= ~keep
            bound = bound.masked_fill(~keep, NEG_INF)

        if banned:
            hit = [self.index[b] for b in banned if b in self.index]
            if hit:
                ix = torch.tensor(hit, device=self.device)
                bound[ix] = NEG_INF     # sorts them last
                ban_mask[ix] = True     # and removes them from the scores

        # Only ever visit words that could win. Sorting the whole vocabulary and
        # relying on the -inf bound to skip the rest still pushes a full chunk
        # of masked-out words through the model: a 3-word admissible set cost
        # MORE than an unfiltered decision (measured 87s vs 32s on CPU) because
        # both scored 512 rows. Restricting the scan pool to the admissible set
        # makes a small set genuinely cheap, which is the common case once the
        # feedback filter bites.
        if allowed_idx is not None:
            pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
            pool = pool[~ban_mask[pool]]
            if pool.numel() == 0:                    # everything banned
                pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
        else:
            pool = torch.arange(self.n, device=self.device)
        order = pool[torch.argsort(bound[pool], descending=True)]
        n_scan = int(order.numel())

        best_i, best_s, second = -1, NEG_INF, NEG_INF
        n_chunks = 0
        for s in range(0, n_scan, self.chunk):
            rows = order[s:s + self.chunk]
            if bound[rows[0]] == NEG_INF:
                break                                    # all remaining banned
            if best_s >= bound[rows[0]].item():
                break                # no unvisited word can beat the incumbent
            sc = self._score_rows(model, cache, lp0, plen, rows.cpu())
            # A banned word can still land in the tail of an otherwise-live
            # chunk. Masking the bound alone is not enough -- the score has to
            # be masked too, or the ban is silently ignored.
            sc = sc.masked_fill(ban_mask[rows], NEG_INF)
            n_chunks += 1
            top2 = torch.topk(sc, min(2, sc.numel()))
            if top2.values[0].item() > best_s:
                second = max(second, best_s)
                if top2.values.numel() > 1:
                    second = max(second, top2.values[1].item())
                best_s = top2.values[0].item()
                best_i = rows[top2.indices[0]].item()
            elif top2.values[0].item() > second:
                second = top2.values[0].item()

        margin = None if second == NEG_INF else best_s - second
        return self.words[best_i], best_s, n_chunks, margin

    @torch.no_grad()
    def rank_of(self, scores, words):
        """1-based ranks of `words` under a full `score_all` vector."""
        order = torch.argsort(scores, descending=True)
        pos = torch.empty_like(order)
        pos[order] = torch.arange(order.numel(), device=order.device)
        return {w: int(pos[self.index[w]].item()) + 1
                for w in words if w in self.index}

    # -- feedback-consistent selection (Phase 7) ----------------------------
    @torch.no_grad()
    def select(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Pick a word, optionally restricted to an admissible subset.

        `allowed_idx` is an index array into `self.words`. It is intended to
        carry the FEEDBACK-CONSISTENT set: the legal words that would have
        produced exactly the feedback already observed. That set is a pure
        function of the prompt (history + the public word list) -- it never
        touches the answer list -- so restricting to it is the same class of
        move as restricting to legal words.

        Returns a dict, not a tuple, so callers can record why a word was
        chosen. The distinction matters for interpreting results:

            forced       len(allowed) == 1. The filter determined the word.
                         The model contributed nothing and this must NOT be
                         counted as a model decision.
            model_chosen len(allowed) > 1. The model ranked the admissible
                         words and picked one.

        Reporting these together would let a decoder improvement masquerade as
        a model improvement.
        """
        n_allowed = self.n if allowed_idx is None else int(len(allowed_idx))

        if allowed_idx is not None and n_allowed == 0:
            # Cannot happen while the answer is legal and the feedback honest,
            # but degrade to the full pool rather than crash a 4-hour run.
            allowed_idx, n_allowed = None, self.n

        if allowed_idx is not None and n_allowed == 1:
            i = int(allowed_idx[0])
            return {"word": self.words[i], "score": None, "n_chunks": 0,
                    "margin": None, "n_allowed": 1, "forced": True,
                    "model_chosen": False}

        w, s, nch, margin = self.argmax(model, prompt, banned=banned,
                                        allowed_idx=allowed_idx, prune=prune)
        return {"word": w, "score": s, "n_chunks": nch, "margin": margin,
                "n_allowed": n_allowed, "forced": False, "model_chosen": True}

    # -- correctness --------------------------------------------------------
    @torch.no_grad()
    def self_test(self, model, prompt, n_probe=32, atol=None):
        """Check cache-reuse scoring against naive full-sequence scoring.

        This is the guard on the KV-cache shim above. A shim that mishandles a
        transformers version produces essentially random scores -- wrong by
        many nats, with the ordering destroyed. That is the failure this must
        catch, and it is enormous.

        What it must NOT flag is float16 rounding. The two paths run different
        matmul shapes (one sequence of length P+L, versus a batch of L-token
        rows against an expanded P-token cache), so fp16 reduction order
        differs and summed log-probs disagree at the 0.01-0.1 nat level. That
        is arithmetic noise, not a broken cache.

        So the test is two-sided:
          * absolute deviation under a dtype-aware tolerance, and
          * the two score vectors still rank the probe words the same way
            (correlation ~1). A broken shim cannot preserve the ranking.

        Note this discrepancy is a *validation* artifact only. Every one of the
        12,972 words is scored through the same fast path, so the ranking the
        argmax is read off is internally consistent.

        Probe indices are spread evenly across the whole word list, not taken
        from the front: if the prompt cache were mutated in place by the first
        chunk's forward pass, only words in *later* chunks would be wrong, and
        a probe drawn from the front would miss it entirely.
        """
        dtype = next(model.parameters()).dtype
        if atol is None:
            atol = 0.02 if dtype in (torch.float32, torch.float64) else 0.40

        fast = self.score_all(model, prompt)
        p_ids = self.tok(prompt, return_tensors="pt",
                         add_special_tokens=False)["input_ids"].to(self.device)
        n_probe = min(n_probe, self.n)
        probe = [int(round(i * (self.n - 1) / max(n_probe - 1, 1)))
                 for i in range(n_probe)]

        got, ref_all = [], []
        for i in probe:
            ids = self.tokens[i][self.mask[i] > 0].unsqueeze(0)
            full = torch.cat([p_ids, ids], dim=1)
            logits = model(input_ids=full, use_cache=False).logits[0].float()
            lp = F.log_softmax(logits, dim=-1)
            start = p_ids.shape[1] - 1
            ref = sum(lp[start + j, ids[0, j]].item() for j in range(ids.shape[1]))
            if self.length_normalise:
                ref /= ids.shape[1]
            ref_all.append(ref)
            got.append(fast[i].item())

        a = np.asarray(got, dtype=np.float64)
        b = np.asarray(ref_all, dtype=np.float64)
        worst = float(np.max(np.abs(a - b)))
        spread = float(b.max() - b.min())
        corr = (float(np.corrcoef(a, b)[0, 1])
                if a.std() > 1e-9 and b.std() > 1e-9 else 1.0)

        assert corr > 0.999, (
            f"constrained scorer does not preserve the ranking of the naive "
            f"scorer (corr={corr:.6f}). The KV-cache shim in expand_cache() "
            f"does not match this transformers version -- do not trust "
            f"constrained results.")
        assert worst < atol, (
            f"constrained scorer disagrees with naive scoring by {worst:.4f} "
            f"nats (tol {atol} for dtype {dtype}), across a probe score spread "
            f"of {spread:.1f} nats. Ranking is preserved (corr={corr:.6f}), so "
            f"this looks like arithmetic noise rather than a broken cache -- "
            f"but it is larger than expected. Investigate before trusting the "
            f"numbers.")
        return {"max_abs_dev": worst, "corr": corr, "spread": spread,
                "atol": atol, "dtype": str(dtype), "n_probe": n_probe}

    @torch.no_grad()
    def verify_against_full(self, model, prompt):
        """Assert the pruned argmax equals the unpruned argmax."""
        full = self.score_all(model, prompt)
        w_full = self.words[int(full.argmax().item())]
        w_prune, _, n_chunks, _ = self.argmax(model, prompt, prune=True)
        assert w_full == w_prune, (
            f"pruning changed the answer: full={w_full} pruned={w_prune}. "
            f"The branch-and-bound bound is wrong.")
        return w_full, n_chunks


# ---------------------------------------------------------------------------
class HardModeFilter:
    """The legal words consistent with every piece of feedback received.

    A word `w` is admissible iff, for every past guess `g` with observed
    pattern `p`, `feedback_code(g, w) == p`. That is precisely the set a
    hard-mode Wordle player can compute from their own board.

    WHAT THIS IS NOT: the candidate set. Two sets are easy to conflate and the
    difference is the whole justification --

        candidate set   answers consistent with feedback   pool = 2,315 answers
                        -> uses the ANSWER LIST, privileged, never used here
        hard-mode set   legal guesses consistent with it   pool = 12,972 legal
                        -> a pure function of the prompt + the public word list

    The model is never shown this set, its size, the answer, or the answer
    list. It is a decoder-side restriction on which words may be selected,
    exactly like the legal-word constraint.

    Refinement is incremental, so cost is dominated by the first turn and
    collapses immediately after (measured: 12,972 -> ~211 -> ~7 -> ~2).
    """

    def __init__(self, legal_words_lower, feedback_code_fn):
        self.words = list(legal_words_lower)
        self._fb = feedback_code_fn
        self.history = []

    def refine(self, guess, code):
        """Apply one (guess, feedback) pair. `guess` lower-case, `code` int."""
        g = guess.lower()
        self.words = [w for w in self.words if self._fb(g, w) == code]
        self.history.append((g, code))
        return self

    def indices(self, scorer):
        """Index array into `scorer.words` (which are upper-case)."""
        return np.array([scorer.index[w.upper()] for w in self.words
                         if w.upper() in scorer.index], dtype=np.int64)

    def contains(self, word):
        return word.lower() in self.words

    def __len__(self):
        return len(self.words)


# ---------------------------------------------------------------------------
LEGAL_WORDS_SORTED = sorted(LEGAL_GUESSES)
SCORER = None
_VERIFIED = False

def build_scorer(device="cuda"):
    global SCORER
    if SCORER is None:
        t0 = time.perf_counter()
        SCORER = LegalWordScorer(TOKENIZER, LEGAL_WORDS_SORTED, device=device,
                                 chunk=CONSTRAINED_CHUNK,
                                 length_normalise=LENGTH_NORMALISE)
        print(f"scorer over {SCORER.n} words in {time.perf_counter()-t0:.1f}s")
    return SCORER

def verify_scorer(model):
    """Fatal on failure. Every check here has caught a real bug at least once."""
    global _VERIFIED
    sc = build_scorer()
    probe = GameState("CRANE", VOCAB.n_answers).prompt(1, MAX_GUESSES)
    d = sc.self_test(model, probe)
    print(f"  cache vs naive [{d['dtype']}]: max|delta|={d['max_abs_dev']:.4f} "
          f"nats (tol {d['atol']}), corr={d['corr']:.6f}  OK")
    w, nch = sc.verify_against_full(model, probe)
    print(f"  pruned argmax == full argmax ({w})  OK")

    # ban must mask SCORES, not just the pruning bound
    full = sc.score_all(model, probe)
    want = [sc.words[i] for i in torch.argsort(full, descending=True)[:3].tolist()]
    got, ban = [], []
    for _ in range(3):
        r = sc.select(model, probe, banned=ban or None)
        assert r["word"] not in ban
        got.append(r["word"]); ban.append(r["word"])
    assert got == want, f"banning broke the ranking: {got} != {want}"
    print(f"  sequential banning walks the global ranking {got}  OK")

    # allowed_idx must ALSO mask scores: pick a subset excluding the global best
    gbest = sc.words[int(full.argmax())]
    sub = [i for i in range(sc.n) if sc.words[i] != gbest][:400]
    idx = np.array(sub, dtype=np.int64)
    r = sc.select(model, probe, allowed_idx=idx)
    assert r["word"] != gbest, "allowed_idx leaked a non-admissible word"
    best_sub = max(sub, key=lambda i: full[i].item())
    assert r["word"] == sc.words[best_sub], "allowed_idx did not pick the best admissible"
    print(f"  allowed_idx excludes {gbest}, returns best admissible "
          f"({r['word']})  OK")

    one = np.array([sc.index[gbest]], dtype=np.int64)
    r1 = sc.select(model, probe, allowed_idx=one)
    assert r1["forced"] and not r1["model_chosen"] and r1["n_chunks"] == 0
    print(f"  single admissible word -> forced, no model call  OK")
    _VERIFIED = True
    return d

print("decoders ready.")

---
# 7. Evaluation harness

One `GameState` per game carries its own `HardModeFilter`. Turn 1 is a no-op
(the whole legal pool); after that the filter refines on the observed feedback.

An eval-only invariant asserts the true answer stays admissible. It is a check
on the filter, never an input to a decision.

In [ ]:
import re
from peft import PeftModel

WORD_RE = re.compile(r"^[A-Za-z]{5}$")

def extract_guess(text):
    line = text.strip().split("\n")[0]
    toks = [t.strip(".,:;!?\"'()[]*_-") for t in line.split()]
    toks = [t for t in toks if t]
    if not toks:
        return None, "invalid_format"
    if WORD_RE.match(toks[0]):
        return toks[0].upper(), "ok"
    return None, "invalid_format"

class GameState:
    __slots__ = ("answer", "history", "cands", "guesses", "patterns", "remaining",
                 "statuses", "forced", "n_allowed", "done", "solved", "mode",
                 "filt", "win_forced")
    def __init__(self, answer, n_answers, mode="legal", filt=None):
        self.answer, self.mode, self.filt = answer, mode, filt
        self.history = []
        self.cands = np.arange(n_answers, dtype=np.int32)
        self.guesses, self.patterns, self.remaining = [], [], []
        self.statuses, self.forced, self.n_allowed = [], [], []
        self.done = self.solved = False
        self.win_forced = None
    def prompt(self, turn, max_guesses):
        h = [(g.lower(), p) for g, p in self.history]
        return render_prompt(turn=turn, history=h,
                             constraints=derive_constraints(h),
                             n_candidates=len(self.cands),
                             guesses_remaining=max_guesses - turn + 1,
                             max_guesses=max_guesses,
                             candidates=None, show_candidate_count=False)

def _apply(g, word, status, turn, max_guesses, forced=None, n_allowed=None):
    g.statuses.append(status); g.forced.append(forced); g.n_allowed.append(n_allowed)
    if status != "ok":
        g.guesses.append(word or "<INVALID>"); g.patterns.append(None)
        g.remaining.append(int(len(g.cands)))
    else:
        code_ = feedback_code(word.lower(), g.answer.lower())
        g.cands = BUNDLE.fb.filter_indices(g.cands, word.lower(), code_)
        if g.filt is not None:
            g.filt.refine(word.lower(), code_)
            assert g.filt.contains(g.answer), (
                f"INVARIANT BROKEN: answer {g.answer} left the admissible set")
        g.history.append((word, code_to_pattern(code_)))
        g.guesses.append(word); g.patterns.append(code_to_pattern(code_))
        g.remaining.append(int(len(g.cands)))
        if code_ == ALL_GREEN:
            g.solved = g.done = True
            g.win_forced = forced
    if turn == max_guesses and not g.solved:
        g.done = True

@torch.no_grad()
def play_games(model, answers, decoder="legal", ban_repeats=False, scorer=None,
               max_guesses=MAX_GUESSES, batch=EVAL_BATCH, log_every_games=60):
    assert decoder in ("unconstrained", "legal", "consistent", "adaptive")
    model.eval()
    tok = TOKENIZER; old = tok.padding_side; tok.padding_side = "left"
    games = [GameState(a, VOCAB.n_answers, mode=decoder,
                       filt=HardModeFilter(LEGAL_LOWER, feedback_code)
                       if decoder in ("consistent", "adaptive") else None)
             for a in answers]
    t0 = time.perf_counter()
    for turn in range(1, max_guesses + 1):
        active = [g for g in games if not g.done]
        if not active:
            break
        if decoder == "unconstrained":
            for s in range(0, len(active), batch):
                ch = active[s:s+batch]
                enc = tok([g.prompt(turn, max_guesses) for g in ch],
                          return_tensors="pt", padding=True, truncation=True,
                          max_length=MAX_SEQ_LEN).to(model.device)
                out = model.generate(**enc, max_new_tokens=GEN_MAX_NEW_TOKENS,
                                     do_sample=False, temperature=None,
                                     top_p=None, top_k=None,
                                     pad_token_id=tok.pad_token_id)
                for g, txt in zip(ch, tok.batch_decode(
                        out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
                    w, st = extract_guess(txt)
                    if w is not None and w not in LEGAL_GUESSES:
                        st = "invalid_word"
                    _apply(g, w, st, turn, max_guesses)
        else:
            for i, g in enumerate(active):
                p = g.prompt(turn, max_guesses)
                banned = list(dict.fromkeys(g.guesses)) if ban_repeats else None
                allowed = None
                if decoder == "consistent":
                    allowed = g.filt.indices(scorer)
                elif decoder == "adaptive":
                    # probe freely while the admissible set is large; become
                    # consistent once the state is nearly resolved
                    if len(g.filt) <= ADAPTIVE_THRESHOLD:
                        allowed = g.filt.indices(scorer)
                r = scorer.select(model, p, banned=banned, allowed_idx=allowed,
                                  prune=CONSTRAINED_PRUNE)
                _apply(g, r["word"], "ok", turn, max_guesses,
                       forced=r["forced"], n_allowed=r["n_allowed"])
                if log_every_games and (i+1) % log_every_games == 0:
                    print(f"      turn {turn}: {i+1}/{len(active)}  "
                          f"{time.perf_counter()-t0:.0f}s", flush=True)
        print(f"  turn {turn}: {sum(1 for g in games if g.done)}/{len(games)} "
              f"done ({time.perf_counter()-t0:.0f}s)", flush=True)
    tok.padding_side = old
    return games

def score_games(games, label, decoder, ban, max_guesses=MAX_GUESSES):
    n = len(games)
    scores = [len(g.guesses) if g.solved else max_guesses+1 for g in games]
    solved = [len(g.guesses) for g in games if g.solved]
    dist = {k: sum(1 for g in games if g.solved and len(g.guesses) == k)
            for k in range(1, max_guesses+1)}
    cum = lambda m: 100*sum(dist[i] for i in range(1, m+1))/n
    st = [s for g in games for s in g.statuses]
    rate = lambda x: 100*sum(1 for s in st if s == x)/max(len(st), 1)
    rep = sum(1 for g in games if len(g.guesses) != len(set(g.guesses)))
    hv = tv = 0
    for g in games:
        seen = []
        for gu, pat in zip(g.guesses, g.patterns):
            if pat is None: continue
            tv += 1
            greens = {}
            for pg, pp in seen:
                for i, (c, t) in enumerate(zip(pg, pp)):
                    if t == "G": greens[i] = c
            if any(gu[i] != c for i, c in greens.items()): hv += 1
            seen.append((gu, pat))
    dec = [f for g in games for f in g.forced if f is not None]
    n_forced = sum(1 for f in dec if f)
    wins_forced = sum(1 for g in games if g.solved and g.win_forced is True)
    wins_model = sum(1 for g in games if g.solved and g.win_forced is False)
    na = [x for g in games for x in g.n_allowed if x is not None]
    return {
        "model": label, "decoder": decoder, "ban_repeats": ban, "n_games": n,
        "mean_failures_as_7": round(sum(scores)/n, 4),
        "mean_solved_only": round(sum(solved)/len(solved), 4) if solved else None,
        "median": float(np.median(solved)) if solved else None,
        "max": max(solved) if solved else None,
        "pct_le3": round(cum(3), 2), "pct_le4": round(cum(4), 2),
        "pct_le5": round(cum(5), 2), "distribution": dist,
        "failures": n-len(solved),
        "failure_rate_pct": round(100*(n-len(solved))/n, 2),
        "solved": len(solved),
        "invalid_format_rate_pct": round(rate("invalid_format"), 2),
        "invalid_word_rate_pct": round(rate("invalid_word"), 2),
        "repeated_guess_game_rate_pct": round(100*rep/n, 2),
        "hard_mode_violation_pct": round(100*hv/max(tv, 1), 2),
        # the split that stops a decoder win looking like a model win
        "decisions": len(dec), "forced_decisions": n_forced,
        "model_decisions": len(dec)-n_forced,
        "forced_decision_pct": round(100*n_forced/max(len(dec), 1), 2),
        "wins_forced": wins_forced, "wins_model_chosen": wins_model,
        "median_n_allowed": float(np.median(na)) if na else None,
        "avg_candidates_after_turn": {
            str(t): round(float(np.mean([g.remaining[t-1] for g in games
                                         if len(g.remaining) >= t])), 2)
            for t in range(1, max_guesses+1)
            if any(len(g.remaining) >= t for g in games)},
    }

def load_adapter(path):
    m = PeftModel.from_pretrained(load_base_model(), path)
    m.config.use_cache = True
    return m.eval().cuda()

print("harness ready. candidate list/count/answer shown: False (never)")

---
# 8. Result containers — with resume

`results.json` is reloaded if present, so a re-run skips evaluations that
already completed rather than repeating them.

In [ ]:
EVAL_ROWS, GAMES_ALL, BASELINES = {}, {}, []
K1 = {}; K1_ROWS = []

RESULTS_PATH = os.path.join(RESULTS_ROOT, "results.json")
if os.path.exists(RESULTS_PATH):
    try:
        prev = json.load(open(RESULTS_PATH, encoding="utf-8"))
        EVAL_ROWS = prev.get("eval_rows", {})
        BASELINES = prev.get("classical_baselines", [])
        K1 = prev.get("k1_probe", {})
        print(f"RESUMED: {sorted(EVAL_ROWS)}")
    except Exception as e:
        print(f"could not resume ({e}); starting fresh")

def key(dec, ban):
    return f"{dec}__{'ban' if ban else 'noban'}"

def save_state(tag=""):
    json.dump({
        "phase": 7, "adapter": ADAPTER_NAME, "model_name": MODEL_NAME,
        "training": TRAIN_CONFIG, "data_stats": DATA_STATS,
        "eval_rows": EVAL_ROWS, "classical_baselines": BASELINES,
        "k1_probe": K1, "phase6_reference": PHASE6,
        "eval_settings": {
            "legal_words": len(LEGAL_WORDS_SORTED), "chunk": CONSTRAINED_CHUNK,
            "exact_pruning": CONSTRAINED_PRUNE,
            "candidate_list_shown": False, "candidate_count_shown": False,
            "answer_shown": False,
            "consistent_decoder": "legal words w with feedback_code(g,w)==observed "
                                  "for every past guess g; a function of the "
                                  "prompt, never the answer list"},
        "environment": {"torch": torch.__version__,
                        "transformers": transformers.__version__,
                        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
                        "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
    }, open(RESULTS_PATH, "w", encoding="utf-8"), indent=2, default=str)
    if tag: print(f"    [saved: {tag}]", flush=True)

save_state("init")
print(f"results -> {RESULTS_PATH}")

---
# 9. The three-way comparison

In [ ]:
if RUN_EVALUATION:
    MODEL = None
    for dec, ban in EVAL_MATRIX:
        k = key(dec, ban)
        if k in EVAL_ROWS:
            print(f"{k}: already done (resumed), skipping"); continue
        print("=" * 66)
        print(f"EVALUATING  decoder={dec}  ban_repeats={ban}")
        print("=" * 66)
        if MODEL is None:
            MODEL = load_adapter(ADAPTER_DIR)
        sc = None
        if dec != "unconstrained":
            sc = build_scorer()
            if not _VERIFIED:
                verify_scorer(MODEL)
        t0 = time.perf_counter()
        gs = play_games(MODEL, VAL_ANSWERS, decoder=dec, ban_repeats=ban, scorer=sc)
        row = score_games(gs, f"{ADAPTER_NAME} [{dec}/{'ban' if ban else 'noban'}]",
                          dec, ban)
        row["eval_seconds"] = round(time.perf_counter()-t0, 1)
        GAMES_ALL[k] = gs; EVAL_ROWS[k] = row
        print(f"  mean={row['mean_failures_as_7']:.4f}  "
              f"fail={row['failure_rate_pct']:.1f}%  solved={row['solved']}  "
              f"invalid={row['invalid_word_rate_pct']:.1f}%  "
              f"repeat={row['repeated_guess_game_rate_pct']:.1f}%  "
              f"hardviol={row['hard_mode_violation_pct']:.1f}%")
        if dec in ("consistent", "adaptive"):
            print(f"  forced {row['forced_decisions']}/{row['decisions']} "
                  f"({row['forced_decision_pct']:.1f}%)   "
                  f"wins: forced {row['wins_forced']} / "
                  f"model-chosen {row['wins_model_chosen']}")
        if dec == "legal" and not ban:
            d = row["mean_failures_as_7"] - PHASE6["endgame_legal_noban"]
            print(f"  PHASE 6 REPRODUCTION: {row['mean_failures_as_7']:.4f} vs "
                  f"{PHASE6['endgame_legal_noban']:.4f} ({d:+.4f})"
                  f"{'  OK' if abs(d) < 0.08 else '  <-- MISMATCH'}")
        print(); save_state(k)
    if MODEL is not None:
        del MODEL; torch.cuda.empty_cache()
else:
    print("evaluation skipped")

---
# 10. Classical baselines

In [ ]:
if RUN_BASELINES and not BASELINES:
    from tree_search import TreeSearchConfig, TreeSearchSolver
    cfgc = SolverConfig(max_guesses=MAX_GUESSES, seed=SEED, guess_pool="full")
    lower = [a.lower() for a in VAL_ANSWERS]
    def summarize(label, games, n=len(VAL_ANSWERS)):
        sc_ = [g.score for g in games]; so = [g.n_guesses for g in games if g.solved]
        dist = {k: sum(1 for g in games if g.solved and g.n_guesses == k) for k in range(1, 7)}
        cum = lambda m: 100*sum(dist[i] for i in range(1, m+1))/n
        return {"model": label, "n_games": n,
                "mean_failures_as_7": round(sum(sc_)/n, 4),
                "pct_le3": round(cum(3), 2), "pct_le4": round(cum(4), 2),
                "failure_rate_pct": round(100*(n-len(so))/n, 2),
                "solved": len(so), "classical": True}
    for lbl in ["random", "frequency", "entropy"]:
        sv = make_solver(lbl, BUNDLE.fb, cfgc, BUNDLE.model); sv.reset()
        op = sv.opening_guess() if sv.deterministic else None
        BASELINES.append(summarize(lbl, [play_game(sv, a, first_guess=op) for a in lower]))
        print(f"  {lbl:<12} {BASELINES[-1]['mean_failures_as_7']:.4f}")
    tc = TreeSearchConfig(depth=6, top_k=100, endgame_top_k=60,
                          endgame_threshold=10, opening_guess="salet",
                          max_guesses=MAX_GUESSES)
    sv = TreeSearchSolver(BUNDLE.fb, cfgc, tc)
    BASELINES.append(summarize("tree_salet", [play_game(sv, a, first_guess="salet") for a in lower]))
    print(f"  {'tree_salet':<12} {BASELINES[-1]['mean_failures_as_7']:.4f}")
    save_state("baselines")
else:
    print("baselines skipped or resumed")

---
# 11. k=1 probe — unfiltered

Deliberately run **without** the consistency filter. This measures what the
*model* knows, and stays comparable to Phase 5/6 (20.0% → 27.5%). Filtering it
would measure the decoder instead.

In [ ]:
def terminal_states(answers, k_target=1, cap=K1_MAX_STATES):
    cfgc = SolverConfig(max_guesses=MAX_GUESSES, seed=SEED, guess_pool="full")
    sv = make_solver("entropy", BUNDLE.fb, cfgc, BUNDLE.model); sv.reset()
    opener = sv.opening_guess()
    out = []
    for ans in answers:
        low = ans.lower(); cands = np.arange(VOCAB.n_answers, dtype=np.int32)
        hist = []
        for turn in range(1, MAX_GUESSES+1):
            n = int(len(cands))
            if n == k_target:
                out.append({"answer": ans, "turn": turn,
                            "candidates": [VOCAB.answers[i].upper() for i in cands],
                            "prompt": render_prompt(
                                turn=turn, history=list(hist),
                                constraints=derive_constraints(hist), n_candidates=n,
                                guesses_remaining=MAX_GUESSES-turn+1,
                                max_guesses=MAX_GUESSES, candidates=None,
                                show_candidate_count=False)})
                break
            if n <= 1: break
            g = opener if turn == 1 else sv.choose(cands, turn)
            c = feedback_code(g, low); hist.append((g, code_to_pattern(c)))
            cands = BUNDLE.fb.filter_indices(cands, g, c)
            if c == ALL_GREEN: break
        if len(out) >= cap: break
    return out

if RUN_K1_PROBE and "tree_salet_endgame" not in K1:
    sc = build_scorer()
    M = load_adapter(ADAPTER_DIR)
    if not _VERIFIED:
        verify_scorer(M)
    TRAIN_TARGETS = set()
    for f in ("data/tree_salet/train.jsonl", "data/tree_salet_endgame/train.jsonl"):
        p = os.path.join(SFT_DIR, f)
        if os.path.exists(p):
            for l in open(p, encoding="utf-8"):
                TRAIN_TARGETS.add(json.loads(l)["completion"].upper())
    ST = terminal_states(VAL_ANSWERS, 1)
    print(f"{len(ST)} k=1 probe states")
    hit = 0; ranks = []
    t0 = time.perf_counter()
    for i, s in enumerate(ST):
        sc_all = sc.score_all(M, s["prompt"])
        top1 = sc.words[int(sc_all.argmax().item())]
        r = sc.rank_of(sc_all, [s["answer"]]).get(s["answer"])
        hit += int(top1 == s["answer"]); ranks.append(r)
        K1_ROWS.append({"answer": s["answer"], "top1": top1,
                        "correct": top1 == s["answer"], "rank": r,
                        "in_train_targets": s["answer"] in TRAIN_TARGETS})
        if (i+1) % 10 == 0:
            el = time.perf_counter()-t0
            print(f"  {i+1}/{len(ST)}  {el:.0f}s  ~{el/(i+1)*(len(ST)-i-1):.0f}s left",
                  flush=True)
    K1["tree_salet_endgame"] = {
        "n_states": len(ST), "top1_accuracy_pct": round(100*hit/len(ST), 2),
        "median_rank_of_answer": float(np.median([r for r in ranks if r])),
        "chance_top1_pct": round(100/sc.n, 4)}
    print(f"\n  k=1 top-1 {K1['tree_salet_endgame']['top1_accuracy_pct']}%  "
          f"median rank {K1['tree_salet_endgame']['median_rank_of_answer']}  "
          f"[Phase 6: {PHASE6['k1_top1']}%, rank {PHASE6['k1_median_rank']}]")
    del M; torch.cuda.empty_cache()
    save_state("k1")
else:
    print("k=1 probe skipped or resumed")

---
# 12. Comparison, decomposition, verdict

In [ ]:
import csv

print("=" * 108)
print(f"PHASE 7 - {len(VAL_ANSWERS)} held-out answers")
print("=" * 108)
h = (f"{'run':<40}{'mean':>8}{'fail%':>7}{'solved':>8}{'invalid%':>9}"
     f"{'repeat%':>8}{'hardviol%':>10}{'forced%':>8}")
print(h); print("-"*len(h))
for b in BASELINES:
    print(f"{'classical '+b['model']:<40}{b['mean_failures_as_7']:>8.4f}"
          f"{b['failure_rate_pct']:>7.1f}{b['solved']:>8}{0.0:>9.1f}{0.0:>8.1f}"
          f"{0.0:>10.1f}{'-':>8}")
for k in [key(d, b) for d, b in EVAL_MATRIX]:
    r = EVAL_ROWS.get(k)
    if not r: continue
    fp = (f"{r['forced_decision_pct']:>8.1f}"
          if r["decoder"] in ("consistent", "adaptive") else f"{'-':>8}")
    print(f"{r['model']:<40}{r['mean_failures_as_7']:>8.4f}"
          f"{r['failure_rate_pct']:>7.1f}{r['solved']:>8}"
          f"{r['invalid_word_rate_pct']:>9.1f}{r['repeated_guess_game_rate_pct']:>8.1f}"
          f"{r['hard_mode_violation_pct']:>10.1f}{fp}")

def m(d, b):
    r = EVAL_ROWS.get(key(d, b)); return r["mean_failures_as_7"] if r else None

print("\n" + "="*70); print("DECOMPOSITION"); print("="*70)
u, ln, lb, cn, cb = m("unconstrained", False), m("legal", False), m("legal", True), \
                    m("consistent", False), m("consistent", True)
if u and ln: print(f"  legal-word constraint     : {u:.4f} -> {ln:.4f}  ({ln-u:+.4f})")
if ln and cn: print(f"  + feedback consistency    : {ln:.4f} -> {cn:.4f}  ({cn-ln:+.4f})")
if ln and lb: print(f"  repeat banning (legal)    : {ln:.4f} -> {lb:.4f}  ({lb-ln:+.4f})")
if cn and cb: print(f"  repeat banning (consistent): {cn:.4f} -> {cb:.4f}  ({cb-cn:+.4f})")
ad = m("adaptive", True)
if cb and ad:
    print(f"  adaptive vs always-on      : {cb:.4f} -> {ad:.4f}  ({ad-cb:+.4f})")
    print("    negative => filtering the midgame was costing us the expert's probe")

print("\n" + "="*70); print("WHO WON THE GAMES?"); print("="*70)
for k in (key("consistent", False), key("consistent", True), key("adaptive", True)):
    r = EVAL_ROWS.get(k)
    if not r: continue
    tot = r["wins_forced"] + r["wins_model_chosen"]
    print(f"  {r['model']}")
    print(f"    solved {r['solved']}  =  forced {r['wins_forced']} "
          f"+ model-chosen {r['wins_model_chosen']}")
    if tot:
        print(f"    -> {100*r['wins_forced']/tot:.1f}% of wins were decided by the "
              f"filter alone, not the model")

def verdict():
    ev = []
    best = min([x for x in (cn, cb, ln, lb, m("adaptive", True)) if x is not None],
               default=None)
    if best is None:
        return "INCONCLUSIVE", ["no evaluations completed"]
    rnd = next((b["mean_failures_as_7"] for b in BASELINES if b["model"] == "random"),
               PHASE6["classical_random"])
    ent = next((b["mean_failures_as_7"] for b in BASELINES if b["model"] == "entropy"),
               PHASE6["classical_entropy"])
    ev.append(f"best Phase 7 = {best:.4f}  (Phase 6 best {PHASE6['endgame_legal_ban']:.4f})")
    ev.append(f"classical random {rnd:.4f}, entropy {ent:.4f}")
    r = EVAL_ROWS.get(key("consistent", True)) or EVAL_ROWS.get(key("consistent", False))
    if r:
        tot = r["wins_forced"] + r["wins_model_chosen"]
        ev.append(f"forced decisions {r['forced_decision_pct']:.1f}%; "
                  f"wins forced {r['wins_forced']} vs model {r['wins_model_chosen']}")
        if tot and r["wins_forced"]/tot > 0.6:
            ev.append("most wins came from the filter, not the model")
    if K1.get("tree_salet_endgame"):
        k1 = K1["tree_salet_endgame"]
        ev.append(f"unfiltered k=1 top-1 {k1['top1_accuracy_pct']}% "
                  f"(Phase 6 {PHASE6['k1_top1']}%) - model capability, unchanged by decoding")
    if best <= ent + 0.35: return "SOLVED", ev
    if best < rnd: return "BEATS RANDOM", ev + ["the decoder closed the gap"]
    return "STILL SHORT", ev + ["even a feedback-consistent decoder does not "
                                "reach random elimination"]

V, EV = verdict()
print("\n" + "="*70); print(f"VERDICT: {V}"); print("="*70)
for e in EV: print("  " + e)

FIELDS = ["model", "decoder", "ban_repeats", "n_games", "mean_failures_as_7",
          "failure_rate_pct", "solved", "invalid_word_rate_pct",
          "repeated_guess_game_rate_pct", "hard_mode_violation_pct",
          "decisions", "forced_decisions", "model_decisions",
          "forced_decision_pct", "wins_forced", "wins_model_chosen",
          "median_n_allowed", "pct_le3", "pct_le4", "eval_seconds"]
with open(os.path.join(RESULTS_ROOT, "comparison.csv"), "w", newline="",
          encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS, extrasaction="ignore")
    w.writeheader()
    for r in EVAL_ROWS.values(): w.writerow(r)
if K1_ROWS:
    with open(os.path.join(RESULTS_ROOT, "k1_probe.csv"), "w", newline="",
              encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(K1_ROWS[0].keys()))
        w.writeheader()
        for r in K1_ROWS: w.writerow(r)
json.dump({"verdict": V, "evidence": EV},
          open(os.path.join(RESULTS_ROOT, "verdict.json"), "w", encoding="utf-8"),
          indent=2)
save_state("final")
base = RESULTS_ZIP[:-4]
shutil.make_archive(base, "zip", RESULTS_ROOT)
print(f"\nRESULTS ZIP: {base}.zip "
      f"({os.path.getsize(base+'.zip')/2**20:.2f} MiB) - no weights")
print(f"ADAPTER still at {os.path.join(WORK_DIR, ADAPTER_NAME)} - "
      f"snapshot it as a Dataset if you have not already")